# Module 2 · Lesson 06: Output Parsing

LLMs return **strings**, but your app needs **structured data**.
This lesson covers reliable techniques for parsing LLM outputs.

## What you will learn
1. **JSON parsing** — the most common pattern
2. **Delimiter-based** extraction
3. **Regex** for flexible parsing
4. **Multi-section** parsing
5. **Error handling** and fallback strategies

In [3]:
import os, json, re
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
client = OpenAI()
 
def ask(prompt, system=None, temperature=0, max_tokens=400):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

---
## 1. JSON Parsing — The Go-To Pattern

Ask the model to return JSON, then parse it in Python:

In [4]:
prompt = """"Analyse this product review and return JSON:
 
Review: "The laptop is blazing fast and the screen is gorgoes, but the battery
only lasts 4 hours and it runs hot during gaming."
 
Return JSON with: sentiments (positive/negative/mixed), pros (list), cons (list), rating (1-5)
 
Return ONLY the JSON, no explanation."""
 
raw = ask(prompt)
 
def parse_json(text: str) -> dict:
    """Robustly parse JSON from LLM output."""
    text = text.strip()
    # Remove markdown code blocks
    if "```" in text:
        text = re.sub(r'```(?:json)?\n?', '', text).strip()
    return json.loads(text)
 
parsed = parse_json(raw)
 
display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))

```json
{
  "sentiments": "mixed",
  "pros": [
    "blazing fast",
    "gorgeous screen"
  ],
  "cons": [
    "battery lasts only 4 hours",
    "runs hot during gaming"
  ],
  "rating": 3
}
```

---
## 2. Delimiter-Based Extraction

Use **unique delimiters** to separate sections reliably:

In [5]:
prompt = """Analyse this code and provide feedback in this exact format:
 
---SUMMARY---

One-line summary of the code

---ISSUES---

List eah issue on a new line

---IMPROVED---

The improved code
---END---
 
Code:
```python
def avg(nums):
    return sum(nums) / len(nums)
```
"""

result = ask(prompt)
display(Markdown(result))

---SUMMARY---

Calculates the average of a list of numbers.

---ISSUES---

1. The function does not handle the case where the input list is empty, which would raise a `ZeroDivisionError`.
2. There is no type checking for the input, which could lead to errors if non-numeric values are included in the list.

---IMPROVED---

```python
def avg(nums):
    if not nums:
        raise ValueError("The input list is empty.")
    if not all(isinstance(num, (int, float)) for num in nums):
        raise TypeError("All elements in the input list must be numbers.")
    return sum(nums) / len(nums)
```
---END---

---
## 3. Regex Extraction

When the output format is less predictable, **regex** provides flexibility:

In [6]:
prompt = """List 3 Python packages with their version numbers and a one-line description.
Format each as: package_name (vX.Y.Z) - description"""
 
result =  ask(prompt)
 
print(f"Raw output:\n{result}\n")

Raw output:
1. NumPy (v1.23.5) - A fundamental package for numerical computing in Python, providing support for arrays and matrices along with a collection of mathematical functions.

2. Pandas (v1.5.3) - A powerful data manipulation and analysis library that offers data structures like DataFrames for handling structured data.

3. Matplotlib (v3.6.2) - A comprehensive library for creating static, animated, and interactive visualizations in Python.



In [7]:
pattern = r'(\w+)\s*\(v?([\d.]+)\)\s*[-–]\s*(.+)'
matches = re.findall(pattern, result)
 
if matches:
    print(f"{'Package':<15} {'Version':<10} {'Description'}")
    print("─" * 60)
    for name, version, desc in matches:
        print(f"{name:<15} {version:<10} {desc.strip()}")
else:
    print("⚠️ Regex didn't match — format may have varied")

Package         Version    Description
────────────────────────────────────────────────────────────
NumPy           1.23.5     A fundamental package for numerical computing in Python, providing support for arrays and matrices along with a collection of mathematical functions.
Pandas          1.5.3      A powerful data manipulation and analysis library that offers data structures like DataFrames for handling structured data.
Matplotlib      3.6.2      A comprehensive library for creating static, animated, and interactive visualizations in Python.


---
## 4. OpenAI Structured Output (response_format)

OpenAI offers `response_format={"type": "json_object"}` to **guarantee** valid JSON:

In [8]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {"role":"system", "content":"You analyse text and return JSON"},
        {"role":"user", "content":"Analyse: 'This restaurant has amazing pasta but terrible service.' Return JSON with: sentiment, food_quality (1-5), summary"}
    ],
    response_format = {"type":"json_object"},
    max_tokens = 200
)

In [10]:
parsed = json.loads(response.choices[0].message. content)
print(parsed)

{'sentiment': 'mixed', 'food_quality': 5, 'summary': 'The restaurant offers excellent pasta but has poor service.'}


In [11]:
display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))

```json
{
  "sentiment": "mixed",
  "food_quality": 5,
  "summary": "The restaurant offers excellent pasta but has poor service."
}
```

> 💡 **Best Practice:** Use `response_format={"type": "json_object"}` in production.
> It guarantees valid JSON output — no manual parsing errors!

---
## 5. Error Handling & Fallbacks

---
## 6. Advanced: Structured JSON Game State

Real-world AI apps often need LLMs to return **complex, multi-field JSON** that drives application logic.

This pattern — inspired by community projects like AI dungeon games — uses the **system prompt as a contract** 
that defines the exact JSON schema the model must follow.

> 💡 **Why this matters:** This is the same pattern behind OpenAI's function-calling / tool-use feature (Module 5).
> Mastering JSON contracts now will make agents much easier to build later.

In [12]:
quiz_system = """You are QuizMaster, and AI that generates quiz questions.
 
You MUST always respond with JSON object with this EXACT structure:
{
    "question": "The quiz question text",
    "options": ["A) ...", "B) ...", "C) ...", D) ..."],
    "correct_index": 0,
    "difficulty": "easy | medium | hard",
    "explanation": "Why the correct answer is right (1-2 senteces)",
    "topic": "The topic category"
}
 
Rules:
- Always return exactly 4 options.
- correct_index is 0-based (0=A, 1=B, 2=C, 3=D)
- Return ONLY the JSON, no extra text
"""

In [13]:
raw_quiz = ask("Generate a medium-difficulty question about Python programming",
                system=quiz_system,
                max_tokens = 500)
print(raw_quiz)

{
    "question": "What is the output of the following code: print(type([]) is list)?",
    "options": ["A) True", "B) False", "C) None", "D) Error"],
    "correct_index": 0,
    "difficulty": "medium",
    "explanation": "The expression checks if the type of an empty list is indeed a list, which is true.",
    "topic": "Python Programming"
}


In [15]:
# ── Robust parser with fallbacks ─────────────────────
def robust_parse(text: str) -> dict:
    """Try multiple parsing strategies."""
    # Strategy 1: Direct JSON
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
 
    # Strategy 2: Extract from code blocks
    match = re.search(r'```(?:json)?\s*(.+?)\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
 
    # Strategy 3: Find JSON-like content
    match = re.search(r'\{.+\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
 
    # Strategy 4: Return raw text as fallback
    return {"raw": text, "parse_error": True}
 
# Test with messy outputs
test_cases = [
    '{"name": "test", "value": 42}',
    'Here is the result:\n```json\n{"name": "test"}\n```',
    'The answer is {"result": true} as expected.',
    'No JSON here, just text.',
]
 
for text in test_cases:
    parsed = robust_parse(text)
    status = "Fallback" if parsed.get("parse_error") else "Parsed"
    print(f"{status}: {text[:40]+'...':<45} → {parsed}")

Parsed: {"name": "test", "value": 42}...              → {'name': 'test', 'value': 42}
Parsed: Here is the result:
```json
{"name": "te...   → {'name': 'test'}
Parsed: The answer is {"result": true} as expect...   → {'result': True}
Fallback: No JSON here, just text....                   → {'raw': 'No JSON here, just text.', 'parse_error': True}


In [16]:
# ── Step 3: Parse and validate the structured output ────
 
def validate_quiz(data: dict) -> dict:
    """Validate quiz JSON has all required fields with correct types."""
    errors = []
 
    required = {"question": str, "options": list, "correct_index": int,
                "difficulty": str, "explanation": str, "topic": str}
 
    for field, expected_type in required.items():
        if field not in data:
            errors.append(f"Missing field: '{field}'")
        elif not isinstance(data[field], expected_type):
            errors.append(f"Wrong type for '{field}': expected {expected_type.__name__}")
 
    if "options" in data and isinstance(data["options"], list):
        if len(data["options"]) != 4:
            errors.append(f"Expected 4 options, got {len(data['options'])}")
 
    if "correct_index" in data and isinstance(data["correct_index"], int):
        if data["correct_index"] not in range(4):
            errors.append(f"correct_index {data['correct_index']} out of range 0-3")
 
    if "difficulty" in data and data["difficulty"] not in ("easy", "medium", "hard"):
        errors.append(f"Invalid difficulty: '{data['difficulty']}'")
 
    return {"valid": len(errors) == 0, "errors": errors}
 
# Parse and validate
quiz = robust_parse(raw_quiz)
validation = validate_quiz(quiz)
 
if validation["valid"]:
    print(" Quiz question is valid!\n")
    display(Markdown(f"###  {quiz['question']}\n\n" +
            "\n".join(f"  {opt}" for opt in quiz['options']) +
            f"\n\n**Difficulty:** {quiz['difficulty']} · **Topic:** {quiz['topic']}\n\n" +
            f"**Answer:** {quiz['options'][quiz['correct_index']]}\n\n" +
            f"*{quiz['explanation']}*"))
else:
    print(" Validation failed:")
    for err in validation["errors"]:
        print(f"  • {err}")

 Quiz question is valid!



###  What is the output of the following code: print(type([]) is list)?

  A) True
  B) False
  C) None
  D) Error

**Difficulty:** medium · **Topic:** Python Programming

**Answer:** A) True

*The expression checks if the type of an empty list is indeed a list, which is true.*

In [17]:
# ── Step 4: Build a mini quiz round (3 questions) ────────
 
topics = ["Python data types", "JavaScript closures", "SQL joins"]
 
print("=" * 50)
print("🎮 QUICK QUIZ — 3 Questions")
print("=" * 50)
 
for i, topic in enumerate(topics, 1):
    raw = ask(f"Generate an easy question about {topic}.",
             system=quiz_system, max_tokens=300)
    q = robust_parse(raw)
    v = validate_quiz(q)
 
    if not v["valid"]:
        print(f"\n⚠️ Q{i}: Skipped (invalid format: {v['errors']})")
        continue
 
    print(f"\n📝 Q{i}: {q['question']}")
    for opt in q["options"]:
        print(f"    {opt}")
    print(f"   ✅ Answer: {q['options'][q['correct_index']]}")
    print(f"   💡 {q['explanation']}")
 
print(f"\n{'=' * 50}")
print(f"Generated {len(topics)} structured quiz questions with validation!")

🎮 QUICK QUIZ — 3 Questions

📝 Q1: Which of the following is a mutable data type in Python?
    A) List
    B) Tuple
    C) String
    D) Integer
   ✅ Answer: A) List
   💡 A list is a mutable data type in Python, meaning its contents can be changed after it is created, unlike tuples, strings, and integers which are immutable.

📝 Q2: What is a closure in JavaScript?
    A) A function that retains access to its lexical scope even when the function is executed outside that scope
    B) A method to create a new object
    C) A way to handle asynchronous operations
    D) A type of error handling mechanism
   ✅ Answer: A) A function that retains access to its lexical scope even when the function is executed outside that scope
   💡 A closure allows a function to access variables from its outer scope even after that scope has finished executing, which is a fundamental concept in JavaScript.

📝 Q3: What type of SQL join returns all records from the left table and the matched records from the ri

> 🔑 **Key insight:** The system prompt acts as a *schema contract*. The LLM "agrees" to the format,
> and your code validates compliance. This is exactly how **function-calling** and **tool-use** work 
> in Module 5 — the schema tells the model what shape to return, and your code uses the result.

> **Exercise:** Try modifying the quiz system prompt to add a `"hint"` field. 
> Update `validate_quiz()` to check for it. Does the model comply?

---
## 6. Advanced: Structured JSON Game State

Real-world AI apps often need LLMs to return **complex, multi-field JSON** that drives application logic.

This pattern — inspired by community projects like AI dungeon games — uses the **system prompt as a contract** 
that defines the exact JSON schema the model must follow.

> 💡 **Why this matters:** This is the same pattern behind OpenAI's function-calling / tool-use feature (Module 5).
> Mastering JSON contracts now will make agents much easier to build later.

> 🔑 **Key insight:** The system prompt acts as a *schema contract*. The LLM "agrees" to the format,
> and your code validates compliance. This is exactly how **function-calling** and **tool-use** work 
> in Module 5 — the schema tells the model what shape to return, and your code uses the result.

> **Exercise:** Try modifying the quiz system prompt to add a `"hint"` field. 
> Update `validate_quiz()` to check for it. Does the model comply?

---
## Key Takeaways 📝

| Method | Reliability | Best For |
|--------|-------------|----------|
| `response_format` | ⭐⭐⭐⭐⭐ | Production JSON |
| JSON prompt | ⭐⭐⭐⭐ | Most structured output |
| Delimiters | ⭐⭐⭐ | Multi-section text |
| Regex | ⭐⭐ | Flexible patterns |
| Robust parser | Always | Fallback strategy |

### Production Checklist
1. Use `response_format={"type": "json_object"}` when available
2. Always wrap parsing in try/except
3. Implement fallback strategies
4. Log parse failures for debugging

---
**🎉 Module 2 Complete!**

**Next Module:** `module_03_ai_architecture` — Build production-ready AI applications